# Reconstruct event and comparison-home eligibility

Audit the seven retained events and the stated exclusions for Homes 4 and 17.

**Step:** 06. Event and comparison-home eligibility.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Daily data, hourly monitoring dates, and the seven event dates.

**Outputs:** Reconstructed eligibility rows, event-level counts, and an audit log.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure this reproduction run

Choose the separate output directory. The default is `results/` in the repository folder; rebuilt files in that directory take precedence over supplied inputs. Use a new directory when testing a changed input.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

## Import the required libraries

These imports belong to the active notebook kernel. Install `requirements.txt` in that same environment if an import fails.

In [ ]:
import numpy as np
import pandas as pd
from shared import CODE, DATA, OUTCOMES, load_daily, output_root, dump_json, data_file

## Load daily inputs and define the donor pool

Comparison homes are never-treated homes. Exclude all retained treated homes and Homes 4 and 17, which have recorded works. The two additional Home 4 candidate events are included only to reproduce their exclusion reasons.

In [ ]:
out = output_root(OUTPUT)
daily = load_daily(out)
daily['date'] = pd.to_datetime(daily.date)
events = pd.read_csv(DATA / 'events.csv', parse_dates=['start', 'end'])

treated = set(events.home)
homes = set(daily.home)
donors = homes - treated - {'Home 4', 'Home 17'}   # never-treated comparison pool

# Two Home 4 candidate events, evaluated only to document why they were excluded.
extra = pd.DataFrame([
    dict(event='H04_DMEV_excluded', home='Home 4',
         start=pd.Timestamp('2025-05-08'), end=pd.Timestamp('2025-05-08')),
    dict(event='H04_later_works_excluded', home='Home 4',
         start=pd.Timestamp('2025-07-16'), end=pd.Timestamp('2025-08-25'))])

## Evaluate coverage and baseline concentration

Use the 90 days before work starts and days 1 through 30 after work ends, excluding all construction dates. Require at least 63 pre-period and 21 post-period daily measurements. For PM2.5 only, require a mean of the untransformed baseline daily means no greater than 100 micrograms/m3. Each retained event needs at least five comparisons for every outcome.

In [ ]:
PRE_DAYS, POST_DAYS = 90, 30            # analysis window around the works
MIN_PRE_DAYS, MIN_POST_DAYS = 63, 21     # valid days required per home
MIN_COMPARISONS = 5                      # comparison homes required per outcome
PM25_BASELINE_LIMIT = 100                # ug/m3, PM2.5 only

selections = []   # one row per event x outcome x home
counts = []       # one row per event x outcome
all_events = pd.concat([events[['event', 'home', 'start', 'end']], extra], ignore_index=True)
for e in all_events.itertuples():
    for v in OUTCOMES:
        window = daily[daily.variable.eq(v) & daily.y.notna()]
        pre = window[window.date.between(e.start - pd.Timedelta(days=PRE_DAYS), e.start - pd.Timedelta(days=1))]
        post = window[window.date.between(e.end + pd.Timedelta(days=1), e.end + pd.Timedelta(days=POST_DAYS))]
        pre_n = pre.groupby('home').y.count()
        post_n = post.groupby('home').y.count()
        pre_mean = pre.groupby('home').y.mean()

        selected = []
        for h in sorted(homes):
            n0 = int(pre_n.get(h, 0))
            n1 = int(post_n.get(h, 0))
            mean = pre_mean.get(h, np.nan)
            coverage = n0 >= MIN_PRE_DAYS and n1 >= MIN_POST_DAYS
            baseline = v != 'pm25' or (n0 > 0 and mean <= PM25_BASELINE_LIMIT)
            candidate = h == e.home or h in donors
            include = candidate and coverage and baseline
            selections.append(dict(event=e.event, outcome=v, home=h, pre_n=n0, post_n=n1, pre_mean=mean,
                                   coverage_pass=coverage, baseline_pass=baseline, primary_included=include,
                                   original_pool_included=candidate and coverage))
            if include:
                selected.append(h)

        n_controls = len(set(selected) - {e.home})
        counts.append(dict(event=e.event, outcome=v, treated_home=e.home,
                           pre_n=int(pre_n.get(e.home, 0)), post_n=int(post_n.get(e.home, 0)),
                           n_controls=n_controls,
                           treated_eligible=e.home in selected,
                           metric_eligible=e.home in selected and n_controls >= MIN_COMPARISONS))

## Summarize eligibility

Record eligibility for every event, outcome, and home, including the two excluded Home 4 candidate events.

In [ ]:
selection = pd.DataFrame(selections)
count = pd.DataFrame(counts)
count['all_four_metrics_eligible'] = count.event.map(count.groupby('event').metric_eligible.all())

## Verify the specific event exclusions

Home 4's dMEV event has 89 valid baseline days and only 16 follow-up days. Its later works have 68 and 30 days but only four eligible PM2.5 comparisons. Home 17's last valid hourly observation is May 25, 2025, before its first recorded work on July 18. Monitoring end is an hourly date, not the date of the last complete daily mean.

In [ ]:
h4 = count[count.event.eq('H04_DMEV_excluded') & count.outcome.eq('temp')].iloc[0]
later = count[count.event.eq('H04_later_works_excluded') & count.outcome.eq('pm25')].iloc[0]

# Home 17 is sensor 45. Monitoring end in Table S1 is the last valid hour,
# not the last day with at least 18 valid hours.
hourly = pd.read_csv(data_file('hourly_primary_23.csv.gz', out), usecols=['sensor', 'time', 'value'])
times = pd.to_datetime(hourly.loc[hourly.sensor.eq(45) & hourly.value.notna(), 'time'], utc=True)
last = pd.Timestamp(times.dt.tz_convert('Europe/London').max().date())

assert (h4.pre_n, h4.post_n) == (89, 16)                              # too few follow-up days
assert (later.pre_n, later.post_n, later.n_controls) == (68, 30, 4)   # too few PM2.5 comparisons
assert last == pd.Timestamp('2025-05-25') and last < pd.Timestamp('2025-07-18')

## Write the eligibility evidence

Save both the home-level decisions and event-level counts. The audit log records the exact failed thresholds, allowing the manuscript's exclusions to be checked without relying on legacy home labels.

In [ ]:
selection.to_csv(out / 'tables/reconstructed_home_event_eligibility.csv', index=False)
count.to_csv(out / 'tables/event_eligibility_counts.csv', index=False)
dump_json(out / 'logs/event_eligibility_checks.json', dict(
    primary_home_outcome_event_rows=int(selection.event.isin(events.event).sum()),
    Home4_DMEV=dict(pre_days=89, post_days=16, required_post=21),
    Home4_later_works=dict(pre_days=68, post_days=30, PM25_controls=4, required_controls=5),
    Home17=dict(last_valid_day=str(last.date()), first_recorded_work='2025-07-18', excluded=True)))
print('Event eligibility and the documented Home 4 and Home 17 exclusions summarized.')

## Interpret the outputs

Review the saved CSV files and JSON checks in the selected output directory. Keep the settings in `analysis_settings.json`
fixed when reproducing this manuscript. If a source-data assertion fails, inspect the named records before changing
any thresholds. `README.md` documents the order of all manuscript stages.